# SIH26060 — PolarOps: Antarctic Digital Twin
## Exploratory Data Analysis & Machine Learning Pipeline

This notebook details the **Exploratory Data Analysis (EDA)**, **Rule-Based Feature Engineering**, **Unsupervised Isolation Forest Anomaly Detection**, and **RandomForest Fuel Forecasting** for the Indian Antarctic research stations **MAITRI** and **BHARATI**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from ml.src.data_loader import load_and_validate_telemetry
from ml.src.feature_engineering import build_full_feature_pipeline, IFOREST_FEATURES, FUEL_REGRESSOR_FEATURES
from ml.src.anomaly_detection import AnomalyDetector
from ml.src.fuel_prediction import FuelPredictor
from ml.src.evaluation import create_chronological_station_splits, evaluate_anomaly_detector
from ml.src.inference import run_ml_inference

print('PolarOps ML modules imported successfully.')

### 1. Dataset Loading & Validation
- Rows: 2,000
- Stations: BHARATI (1,000), MAITRI (1,000)
- Duration: ~14 days (2026-01-01 to 2026-01-14)
- Station sampling interval: 20 minutes (readings alternate every 10 min)

In [2]:
df_raw, report = load_and_validate_telemetry()
print('Validation Status:', report['is_valid'])
print('Shape:', df_raw.shape)
display(df_raw.head())

### 2. Exploratory Telemetry Plots
Telemetry parameters across the 14-day monitoring window:

In [3]:
display(Image('plots/eda_telemetry_overview.png'))

### 3. Feature Engineering & Rule-Based Operational Signals
Constructs station-isolated temporal cyclical encodings, operational threshold margins, cross-sensor consistency checks, rolling statistics, and time-deltas.

In [4]:
df_feat = build_full_feature_pipeline(df_raw, include_future_target=True)
print('Total Engineered Features:', df_feat.shape[1])
print('Isolation Forest Features:', len(IFOREST_FEATURES))
print('Fuel Regressor Features:', len(FUEL_REGRESSOR_FEATURES))
assert 'anomaly' not in IFOREST_FEATURES
assert 'anomaly' not in FUEL_REGRESSOR_FEATURES
print('Safety check: target anomaly label is strictly excluded from features.')

### 4. Anomaly Detection Analysis
Evaluation of unsupervised Isolation Forest predictions against ground truth operational failures (generator overheating, electrical power spikes, sudden fuel loss).

In [5]:
display(Image('plots/anomaly_detection_analysis.png'))

### 5. Fuel Consumption & Forecasting Analysis
RandomForestRegressor predicting 1-hour future fuel levels evaluated against the Persistence Baseline.

In [6]:
display(Image('plots/fuel_prediction_analysis.png'))

### 6. Real-Time Streaming Inference Demonstration
Testing unified inference endpoint `run_ml_inference` ready for Alert Engine ingestion.

In [7]:
sample_reading = df_raw.iloc[500].to_dict()
result = run_ml_inference(sample_reading)
import json
print(json.dumps(result, indent=2))